In [1]:
import sys
import requests
sys.path.append(r'E:\wangzhilin\QuantumGalaxy')
sys.path.append(r'E:\wangzhilin\QuantumGalaxy\server_new\blueprints')
from live_editor.models import bitable_works
import json
from QGI.feishu import base_fetch_func,fetch_tenant_token
from copy import deepcopy
bitapp_auth={'app_id':'cli_a1fc0a192db8100c','app_secret':'nNk1fT8h2eVi8B2ymIFangHTpYrOAJTD'}
import pandas as pd
def build_headers():
    
    token=fetch_tenant_token(bitapp_auth)
    #token='u-3atGsodiN9391_BlvmfeZh1kkwt5hlQxo0G0hgew2Ijc'
    headers = {
        'Authorization':
        'Bearer ' + token,  # your access token
        #'Content-Type': 'application/json; charset=utf-8'
    }
    return headers

In [2]:
import logging
def get_logger(name):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(
        r"E:\wangzhilin\QuantumGalaxy\opt\zone\log.log",
        'a',
        encoding='utf-8')
    fh.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter = logging.Formatter(
        #fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',
        fmt='%(levelname)-8s %(message)s',
        datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    fh.setFormatter(formatter)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger
logger=get_logger('atlas_zone')

In [3]:
def get_files(fold_token):
    logger.info('开始检查文件，文件夹地址为　https://quantumgalaxy.feishu.cn/drive/folder/%s'%fold_token)
    r0=requests.get('http://82.156.248.152:81/fs/fold_meta?token=%s'%fold_token).json()
    
    r1=requests.get('http://82.156.248.152:81/fs/fold_children?fld_token=%s'%fold_token).json()
    if r0['code']==0 and r1['code']==0:
        files=r1['data']['files']
        data={}
        for file in files:
            if file['name'] not in ['电力储能骨干核心表Demo','xx骨干核心表模板及范例']:
            #if file['type']=='bitable' and '核心表' in file['name'] and file['name']!='xx骨干核心表Demo':
            #if True:
                data[file['name']]=file['token']
            else:
                logger.warning('排除该文件，类型:%s,名称:%s'%(file['type'],file['name']))
        num=len(data)
        
        logger.info('检查文件夹成功，共查到%d张多维表格参与联查，如下：'%num)
        for d in data.keys():
            logger.info(d)
        
        return {'code':0,'data':data}
    else:
        logger.info('检查文件夹失败，检查地址')
        return {'code':r0['code'],'data':r0}


In [4]:
def append_df(df,new_row):
    '''["level","body","backbone",""rough"]'''
    df=pd.concat((df,pd.DataFrame(data=[new_row],columns=['level','body','backbone','rough'])))
    return df
def get_mark(backbone_mark):
    if backbone_mark:
        mark=[]
        
        for record in backbone_mark:
            fields=record['fields']
            #print(fields)
            if fields['本骨干标记']:
                mark.append(fields['骨干名'])
        if mark:
            if len(mark)==1:
                return (True,mark[0])

            else:
                return (False,'找到复数勾选骨干：%s'%mark)
        else:
            return (False,'没有勾选骨干标记')
    else:
        return (False,'没有勾选本骨干')
def get_files(fold_token):
    logger.info('开始检查文件，文件夹地址为　https://quantumgalaxy.feishu.cn/drive/folder/%s'%fold_token)
    r0=requests.get('http://82.156.248.152:81/fs/fold_meta?token=%s'%fold_token).json()
    
    r1=requests.get('http://82.156.248.152:81/fs/fold_children?fld_token=%s'%fold_token).json()
    if r0['code']==0 and r1['code']==0:
        files=r1['data']['files']
        data={}
        for file in files:
            if file['name'] not in ['电力储能骨干核心表Demo','xx骨干核心表模板及范例']:
            #if file['type']=='bitable' and '核心表' in file['name'] and file['name']!='xx骨干核心表Demo':
            #if True:
                data[file['name']]=file['token']
            else:
                logger.warning('排除该文件，类型:%s,名称:%s'%(file['type'],file['name']))
        num=len(data)
        
        logger.info('检查文件夹成功，共查到%d张多维表格参与联查，如下：'%num)
        for d in data.keys():
            logger.info(d)
        
        return {'code':0,'data':data}
    else:
        logger.info('检查文件夹失败，检查地址')
        return {'code':r0['code'],'data':r0}
def external_check(internal_result,files,report_df):
    '''对所有表格中externallinks+backbonemark进行检查，其对应的
    external_link结构：  
    'external_links': [{'bitapp_id': 'recNx4LYVO',
    'source': 'reclwpDIsc',
    'target': 'recDC1O4Rf',
    'category': '父节点',
    'checked': True,
    'source_name': '精煤',
    'target_name': '动力煤',
    'external_backbone_name': '电力',
    'external_backbone': '未知'},
    
    A表 有externallink L1,其external_backbone为B，则B表必须有相同sourcenametargaetname的externallink L1，其externalbackbone为A。成功则L1 l2为正确边界(只判断L1在出报告更方便？），否则暴露L1为错误'''
    result=deepcopy(internal_result)
    for token,data in result.items():
        logger.critical('对标记为"%s"的表格的边界进行检查,该表共有%s条外边参与联调'%(data['backbone_mark'],len(data['external_links'])))
        logger.critical(data['backbone_mark'])
        data['error_external_links']=[]
        data['good_external_links']=[]
        #print(data['external_links'])
        for link in data['external_links']:
            linktext=link['source_name']+'--'+link['category']+'->'+link['target_name']
            other_token=link['external_backbone']
            #print(internal_result[other_token])
            #othertoken是在internalcheck时根据file表查到
            get=False
            if other_token in internal_result.keys():
                for other_link in internal_result[other_token]['external_links']:
                #当表格在files里，但权限有问题，该表的token不会出现在internalresult中，需要提示
                    otherlinktext=other_link['source_name']+'--'+other_link['category']+'->'+other_link['target_name']
                    if otherlinktext==linktext:
                        get=True
                        if other_link['external_backbone']==token:
                            try:
                                if link.get('target'):
                                    link['source']=other_link['source']
                                else:
                                    print(link)
                                    print(other_link)
                                    link['target']=other_link['target']
                                data['good_external_links'].append(link)
                                logger.info('星图边界识别成功，在"%s"骨干中查到对应边："%s"'%(link['external_backbone_name'],linktext))
                            except:
                               data['error_external_links'].append(link) 
                               body='找到文本相同的星图边界但起终点有问题，可能是方向写反了或者外骨干标重了，如无法解决联系王之霖。边信息：%s'%linktext
                               logger.error(body)
                               report_df=append_df(report_df,['error',body,data['backbone_mark'],True])
                        else:
                            
                            data['error_external_links'].append(link)
                            body='星图边界识别到对应的边，但对应边的外骨干标记与本骨干不符，对应边的外骨干标记为："%s"，对应边为："%s"'%(link['external_backbone_name'],linktext)
                            logger.error(body)
                            report_df=append_df(report_df,['error',body,data['backbone_mark'],True])
                        break
            else:
                data['error_external_links'].append(link)
                body='星图边界识别失败，该边："%s"所对应的"%s"骨干没有经过内部检查，这是对应骨干而非本边的问题'%(linktext,link['external_backbone_name'])
                logger.error(body)
                report_df=append_df(report_df,['error',body,data['backbone_mark'],True])
            if not get:
                data['error_external_links'].append(link)
                body='星图边界识别失败，该边："%s"在对应"%s"骨干中没找到对应边界'%(linktext,link['external_backbone_name'])
                logger.error(body)
                report_df=append_df(report_df,['error',body,data['backbone_mark'],True])
        logger.critical('完成对标记为"%s"的表格的边界进行检查,该表共有%s条外边参与联调，通过边%s条，冲突边%s条\n'%(data['backbone_mark'],len(data['external_links']),len(data['good_external_links']),len(data['error_external_links'])))
        #print(data['backbone_mark'])
        data.pop('external_links')
    return result,report_df
def internal_check(data1,files,report_df):
    '''对多维表格data进行内部检查，梳理出所有外部节点和其对应边，其中孤立外部节点加入error，多层外部节点加入warning，其他加入external,并把原data中其他数据分类到
    'internal_links':[],'external_links':[],'warning_links':[],'error_nodes':[],'internal_nodes':[],'external_nodes':[],
    加上骨干标记'''
    data=deepcopy(data1)
    result={'internal_links':[],'external_links':[],'warning_links':[],'error_links':[],'error_nodes':[],'internal_nodes':[],'external_nodes':[]}
    r,mark=get_mark(data['backbone_mark'])
    if r:
        logger.critical('本骨干标记：')
        logger.critical(mark)
        result['backbone_mark']=mark
    else:
        body='检查到骨干定义表问题：%s'%mark
        logger.error(body)
        report_df=append_df(report_df,['error',body,mark,True])
        return None,None
    
    #result.update(data)
    for link in data['links']:
        if not link['category'] in ['原材料','储运','父节点','供应','同位','生产','反映']:
            link['error']=True
            result['error_links'].append(link)
            body='该边的关系有误：“%s”'%link['category']
            logger.error(body)
            report_df=append_df(report_df,['error',body,mark,True])
        for o in data['nodes']:
            if o['bitapp_id']==link['target']:
                link['target_name']=o['name']
            if o['bitapp_id']==link['source']:
                link['source_name']=o['name']
        if not (link.get('target_name') and link.get('source_name')):
            link['error']=True
            result['error_links'].append(link)
            body='出现边没有找到对应的起/终点，可能是因为起终点没有勾选粗骨干权限。id:"%s",起点："%s" 终点:"%s"'%(link.get('bitapp_id'),link.get('source_name_ref','unknown'),link.get('target_name_ref','unknown'))
            logger.error(body)
            report_df=append_df(report_df,['error',body,mark,True])
            
    for node in data['nodes']:
        if not node['more_info'].get('external_backbone'):
            result['internal_nodes'].append(node)
        else:
            
            id=node['bitapp_id']
            backbone=node['more_info'].get('external_backbone')

            if backbone:
                backbone=backbone[0]
            check=0
            if not files.get(backbone,None):
                '''该外部节点的‘所属外部骨干’字段与文件目录files比对结果，因此要求字段内骨干名与files名一致，不然认为没有找到外骨干'''
                '''涉及errornode的边移入errorlink'''
                logger.error('“%s”节点没有找到对应的外部骨干：“%s”'%(node['name'],backbone))
                #error
                result['error_nodes'].append(node)
                for link in data['links']:
                    if link['source']==id or link['target']=='id':
                        if link['source']==id :
                            other=link['target']
                            for o in data['nodes']:
                                if o['bitapp_id']==other :
                                    #link['source_name']=node['name']
                                    #link['target_name']=o['name']
                                    link['external_backbone_name']=backbone
                                    link['external_backbone']=files.get(backbone,'未知')
                        else:
                            other=link['source']
                            for o in data['nodes']:
                                if o['bitapp_id']==other :
                                    #link['source_name']=o['name']
                                    #link['target_name']=node['name']
                                    link['external_backbone_name']=backbone
                                    link['external_backbone']=files.get(backbone,'未知')
                        result['error_links'].append(link)
                        body='因“%s”节点没有找到对应外部骨干，与其相连的边也出错,不参与联调：[%s]-(%s)->[%s]'%(node['name'],link['source_name'],link['category'],link['target_name'])
                        logger.error(body)
                        report_df=append_df(report_df,['error',body,mark,True])
                        link['error']=True
            else:
                result['external_nodes'].append(node)

                for link in data['links']:
                    
                        if link['source']==id :
                            
                            #从data中除去该边，只留纯外边
                            other=link['target']
                            for o in data['nodes']:
                                if o['bitapp_id']==other :
                                        
                                        
                                        link['source_name']=node['name']
                                        link['target_name']=o['name']
                                        link['external_backbone_name']=backbone
                                        link['external_backbone']=files.get(backbone,'未知')
                                        if not (o['more_info'].get('external_backbone') or o['category']=='指标'):
                                            #说明另一点为内部
                                            link1=deepcopy(link)
                                            link1.pop('source')
                                            logger.info('检查到与“%s”骨干间的正确星图边界：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name']))
                                            result['external_links'].append(link1)
                                            check=max(check,2)
                                            
                                        else:
                                        
                                            if not link.get('checked'):
                                                result['warning_links'].append(link)
                                                #说明另一点为外部,该边加入warning
                                                body='检查到本骨干之外的,属于“%s”骨干的边，该边不参与冲突检查：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name'])
                                                logger.warning(body)
                                                report_df=append_df(report_df,['warning',body,mark,True])
                                            check=max(check,1)
                                        continue
                            link['checked']=True

                                
                                
                        elif link['target']==id:
                            
                            other=link['source']
                            for o in data['nodes']:
                                if o['bitapp_id']==other:
                                        
                                        
                                        link['source_name']=o['name']
                                        link['target_name']=node['name']
                                        link['external_backbone_name']=backbone
                                        link['external_backbone']=files.get(backbone,'未知')
                                        if not (o['more_info'].get('external_backbone') or o['category']=='指标'):
                                            #说明另一点为内部,该边加入external
                                            link1=deepcopy(link)
                                            link1.pop('target')
                                            logger.info('检查到与“%s”骨干间的正确星图边界：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name']))
                                            result['external_links'].append(link1)
                                            check=max(check,2)
                                            
                                        else:
                                            
                                            if not link.get('checked'):
                                                
                                                result['warning_links'].append(link)
                                                #说明另一点为外部,该边加入warning
                                                body='检查到本骨干之外的,属于“%s”骨干的边，该边不参与冲突检查：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name'])
                                                logger.warning(body)
                                                report_df=append_df(report_df,['warning',body,mark,True])
                                            check=max(check,1)
                                        continue
                            link['checked']=True
                        else:
                            #result['internal_links'].append(link)
                            pass
                            '''所有边均未触发则error 该外部节点为孤立，未连入当前骨干'''
                        
                if check==2:
                    #ok
                    pass
                elif check==1:
                    #warning
                    pass
                else:
                    body='检查到孤立外骨干节点，该点为内部错误，不参与冲突检查：%s'%node['name']
                    logger.error(body)
                    report_df=append_df(report_df,['error',body,mark,True])
                    #error
                    result['error_nodes'].append(node)
                    pass
    for link in data['links']:
                if not link.get('checked'):
                    if not link.get('error'):
                        #没有check表示起终点均不是外部
                        result['internal_links'].append(link)
    return result,report_df
#data1=deepcopy(data)
#internal_check(data1)
def find_common_elements(result,files):
    #检查所有表格里 internal_nodes 的name是否有重复，有的话给出表格token和节点name
    dictionary={}
    for k,v in result.items():
        names=[]
        #if v.get('good_external_nodes'):
        for item in v['internal_nodes']:
        #for item in v['good_external_nodes']:
            if item['category']!='企业':
                names.append(item['name'])
        dictionary[k]=names
    
    common_elements = []
    for key1, lst1 in dictionary.items():
        for key2, lst2 in dictionary.items():
            if key1 != key2:
                for element in lst1:
                    if element in lst2  and not ([element,(key1,key2)]  in common_elements or ([element,(key2,key1)]  in common_elements)):
                        common_elements.append([element,(key1,key2)])
    for i in common_elements:
        tokens=i.pop(1)
        names=({v:k for k,v in files.items()}[tokens[0]],{v:k for k,v in files.items()}[tokens[1]])
        i.append(names)

    return common_elements
def work1(files,report_df):
    logger.info('start')

    
    
    raw_datas={}
    internals={}
    externals={}
    internal_result={}
    errorfile=[]
    #result={'raw':raw_datas,'internal':internals,'external':externals}
    for name,token in files.items():
        '''file:{name,token}'''

        #data=bitable_works(token,subgraph_name='粗骨干图')
        data=bitable_works(token)
        raw_datas[token]=data
        logger.critical('对 “%s”  多维表格进行内部检查'%name)
        if data.get('code'):
            body='对 “%s”  多维表格进行内部检查失败，错误信息：%s\n'%(name,data)
            logger.critical(body)
            report_df=append_df(report_df,['critical',body,name,True])
            errorfile.append(name)
            continue
        r,report_df=internal_check(data,files,report_df)
        if r:
            internal_result[token]=r
            logger.critical('完成对 “%s”  多维表格进行内部检查\n'%name)
        else:
            errorfile.append(name)
            logger.critical('无法对 “%s”  多维表格进行内部检查\n'%name)
    logger.critical('内部检查完成，以下表格出现问题不参与联调:%s\n'%errorfile)
    #for errorname in errorfile:
        #files.pop(errorname)
    return internal_result,report_df
def work2(internal_result,files,report_df):

    logger.critical('开始外部联调，参与表格为：')
    logger.critical(str(list(files.keys()))+'\n')
    
    result,report_df=external_check(internal_result,files,report_df)
    
    '''外部检查'''
        
    return result,report_df
                

In [5]:
files={'区域':'FFb1bDuHkaOwFJsJp2Hc2jPan8f'}

In [6]:
report_df=pd.DataFrame(columns=['level','body','backbone','rough'])
internal_result,report_df=work1(files,report_df)

INFO     start


good table


CRITICAL 对 “区域”  多维表格进行内部检查
CRITICAL 本骨干标记：
CRITICAL 区域
CRITICAL 完成对 “区域”  多维表格进行内部检查

CRITICAL 内部检查完成，以下表格出现问题不参与联调:[]



In [7]:
result,report_df=work2(internal_result,files,report_df)

CRITICAL 开始外部联调，参与表格为：
CRITICAL ['区域']

CRITICAL 对标记为"区域"的表格的边界进行检查,该表共有0条外边参与联调
CRITICAL 区域
CRITICAL 完成对标记为"区域"的表格的边界进行检查,该表共有0条外边参与联调，通过边0条，冲突边0条



In [8]:

vertexs=[]
links={}


nodedic={'产品':'Product','企业':'Company','指标':'Indicator','工艺':'Technique','需求':'Demand'}
linkdic={'原材料':'compose','供应':'supply','父节点':'parent','储运':'carry','生产':'produce','同位':'peer','反映':'measure'}
for file_token,data in result.items():
    print(data['backbone_mark'])
    print('内部节点:%s'%len(data['internal_nodes']))
    print('内部边:%s'%len(data['internal_links']))
    print('外部节点:%s'%len(data['external_nodes']))
    print('外部边:%s'%len(data['good_external_links']))
    for node in data['internal_nodes']:
        post_node=deepcopy(node)
        cate=post_node.pop('category')
        cate_new=nodedic.get(cate,cate)
        post_node['category']=cate_new
        more=post_node.pop('more_info')
        post_node['file_token']=file_token
        post_node['backbone']=data['backbone_mark']
        if more:
            post_node.update(more)
        vertexs.append(post_node)
    for link in data['internal_links']:
        post_link={}
        post_link['source']=link['source']
        post_link['target']=link['target']
        post_link['target_name']=link['target_name']
        post_link['source_name']=link['source_name']
        post_link['category']=linkdic.get(link['category'],link['category'])
        if link.get('more_info'):
            
            post_link.update(link['more_info'])
        key=link['source_name']+'--'+link['category']+'->'+link['target_name']
        if key not in links.keys():
            links[key]=post_link
        else:
            body="“%s”骨干内部重复的边:'%s'"%(data['backbone_mark'],key)
            logger.warning(body)
            report_df=append_df(report_df,['warning',body,data['backbone_mark'],True])
            links[key].update(post_link)
    for link in data['good_external_links']:
        #要找到其节点所在本骨干的recordid，不能直接用link里的id。。
        post_link={}
        post_link['source']=link['source']
        post_link['target']=link['target']
        post_link['target_name']=link['target_name']
        post_link['source_name']=link['source_name']
        post_link['category']=linkdic.get(link['category'],link['category'])
        if link.get('more_info'):
            post_link.update(link['more_info'])
        key=link['source_name']+'--'+link['category']+'->'+link['target_name']
        if key not in links.keys():
            links[key]=post_link
        else:
            #print("dup!:'%s'"%key)
            links[key].update(post_link)
post_result={'vertex':vertexs,'link':list(links.values())}
print(len(post_result['vertex']))
print(len(post_result['link']))
        


区域
内部节点:426
内部边:451
外部节点:0
外部边:0
426
451


In [9]:
from QGI.neoapi import Neo4j
#neo4j+ssc://534ea9b7.databases.neo4j.io:7687
#QuantumGalaxy
#atlas
uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
user = "QG_Editor"
password = "editor"
#neo1=Neo4j(uri,user,password)
#nova
#uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
#user = "neo4j"
#password = "QuantumGalaxy"

neo=Neo4j(uri,user,password)
#neo.simple_query('match (n) detach delete n')

In [10]:

def data2cypher(data):
    cyphers=['match (n) detach delete n']
    from copy import deepcopy
    d=deepcopy(data)
    for v in d['vertex']:
        #merge 应该只有名字？和骨干？公司怎么办
        cate=v.pop('category')
        if cate=='Company' or cate=='Indicator':
            name=v.pop('name')
            backbone=v.pop('backbone')
            
            cypher='merge(n:%s {name:"%s"}) \
                on match set n.backbone=n.backbone+",%s" \
                on create set n.backbone="%s" \
                '%(cate,name,backbone,backbone)
            if v:
                other='with n set '
                
                for key,value in v.items():
                    if type(value)==str:
                        value=value.replace('"', '\\\"').replace("'", "\\\'")
                    other+='n.%s="%s",'%(key,value)
                other=other[:-1]
                cypher+=other
            cyphers.append(cypher)
        else:
            s='{'
            for key,value in v.items():
                s+=key+':'+'"%s",'%value
            s=s[:-1]
            s+='}'
            cypher='merge (n:%s %s)'%(cate,s)
            #print(cypher)
            cyphers.append(cypher)

            
            #cypher='merge (n:%s {name:"%s"}) set n.backbone=n.backbone+",%s"'%(cate,v['name'],v['backbone'])
            #print('try this instead:"%s"'%cypher)
            #try:neo.simple_query(cypher)
            #except:print('failed! pass')
    for l in d['link']:
        #link表中对name和bitapp同时匹配 match (n {name:"",bitapp_id:""})
        #边界是不是不对？eg 畜牧业 牛奶- 食品饮料-乳制品
        sourcename,source,targetname,target,cate=l.pop('source_name'),l.pop('source'),l.pop('target_name'),l.pop('target'),l.pop('category')
        more=''
        if l:
            for k,v in l.items():
                more+=k+':'+'"%s",'%v
            more=more[:-1]
        if cate=='produce':
            cypher='match (n {name:"%s"}),(n1 {name:"%s",bitapp_id:"%s"}) create (n)-[r:%s {%s}]->(n1)'%(sourcename,targetname,target,cate,more) 
        else:
            cypher='match (n {name:"%s",bitapp_id:"%s"}),(n1 {name:"%s",bitapp_id:"%s"}) create (n)-[r:%s {%s}]->(n1)'%(sourcename,source,targetname,target,cate,more)          
        '''                                                                                                                                                                                   -------------------------------------+'
        s=deepcopy(data['vertex'][e['source']])
        s_cate=s.pop('category')
        s1='{'
        for key,value in s.items():
            s1+=key+':'+'"%s",'%value
        s1=s1[:-1]
        s1+='}'
        
        t=deepcopy(data['vertex'][e['target']])
        t_cate=t.pop('category')
        s2='{'
        for key,value in t.items():
            s2+=key+':'+'"%s",'%value
        s2=s2[:-1]
        s2+='}'
        cypher='match (n:%s %s),(n1:%s %s) create (n)-[r:%s]->(n1)'%(s_cate,s1,t_cate,s2,e["rela"])
        '''
        cyphers.append(cypher)
        #print(cypher)
        #neo.simple_query(cypher)
    return cyphers
cyphers=data2cypher(post_result)
print(len(cyphers))

878


In [11]:
r=neo.multi_cypher(cyphers)
r

878

把指标加到数据库

In [12]:
r=neo.read_query('match (n:Indicator) where not n.code is null return n.code,n.name')

In [13]:
r[0].values()

['512480.SH', '半导体ETF']

In [14]:
codes=[(record.values()[0].strip(),record.values()[1].strip()) for record in r]
codes

[('512480.SH', '半导体ETF'),
 ('516620.SH', '影视ETF'),
 ('S2707426', '70城二手住宅价格指数:环比'),
 ('512200.SH', '房地产ETF'),
 ('M0066361', '棉花期货价格'),
 ('S5314678', '粘胶短纤市场价格'),
 ('S5062973', '临沂商场价格指数:服装服饰'),
 ('159625.SZ', '绿色电力ETF'),
 ('515210.SH', '钢铁ETF'),
 ('159667.SZ', '工业母机ETF'),
 ('562500.SH', '机器人ETF'),
 ('159638.SZ', '高端装备ETF'),
 ('159663.SZ', '机床ETF'),
 ('515790.SH', '光伏ETF'),
 ('512980.SH', '传媒ETF'),
 ('S0031645', '伦敦黄金交易所现货价格'),
 ('S5801703', '上海有色金属交易所白银价格'),
 ('S5801706', '上海有色金属交易所铂价格'),
 ('518880.SH', '黄金ETF'),
 ('S0031553', '成品油运输指数'),
 ('BADI.GI', '波罗的海干散货指数'),
 ('S0031554', '原油运输指数'),
 ('T3549529', '波罗的海货运指数'),
 ('159790.SZ', '碳中和ETF'),
 ('513330.SH', '恒生互联网ETF'),
 ('159819.SZ', '云计算ETF'),
 ('512720.SH', '计算机ETF'),
 ('516010.SH', '软件ETF'),
 ('516510.SH', '人工智能ETF'),
 ('159998.SZ', '游戏动漫ETF'),
 ('159996.SZ', '家电ETF'),
 ('512400.SH', '有色金属ETF'),
 ('562800.SH', '稀有金属ETF'),
 ('516150.SH', '稀土ETF基金'),
 ('159928.SZ', '消费ETF'),
 ('S6949763', '电池级碳酸锂价格'),
 ('159840.SZ', '锂电池ETF'),
 ('5169

In [15]:
codelist=[record.values()[0].strip() for record in r]
codelist

['512480.SH',
 '516620.SH',
 'S2707426',
 '512200.SH',
 'M0066361',
 'S5314678',
 'S5062973',
 '159625.SZ',
 '515210.SH',
 '159667.SZ',
 '562500.SH',
 '159638.SZ',
 '159663.SZ',
 '515790.SH',
 '512980.SH',
 'S0031645',
 'S5801703',
 'S5801706',
 '518880.SH',
 'S0031553',
 'BADI.GI',
 'S0031554',
 'T3549529',
 '159790.SZ',
 '513330.SH',
 '159819.SZ',
 '512720.SH',
 '516010.SH',
 '516510.SH',
 '159998.SZ',
 '159996.SZ',
 '512400.SH',
 '562800.SH',
 '516150.SH',
 '159928.SZ',
 'S6949763',
 '159840.SZ',
 '516910.SH',
 '561320.SH',
 '159766.SZ',
 '515220.SH',
 '159755.SZ',
 '515030.SH',
 '516110.SH',
 '513360.SH',
 'M0000612',
 '399997.SZ',
 'S0143880',
 '515170.SH',
 'L.DCE',
 'RU.SHF',
 'B.IPE',
 '159870.SZ',
 '159697.SZ',
 '515880.SH',
 '515050.SH',
 '159732.SZ',
 '159865.SZ',
 '159985.SZ',
 'M0010096',
 'M0001384',
 'M0001380',
 'M0009936',
 '511880.SH',
 '512800.SH',
 '513060.SH',
 '512010.SH',
 '513120.SH',
 '512290.SH',
 '159892.SZ',
 '515120.SH',
 '159647.SZ',
 '159883.SZ',
 '159643

In [16]:
from QGI.mysql import get_mysql
mysql=get_mysql()
    


In [17]:
for code,name in codes:
    if '.' in code:
        #print(name)
        mysql.write_query("insert  into qg_indicator_info (qg_code,name,category,need_process) values ('%s','%s','ticker',1)"%(code,name))
        mysql.write_query('insert  into ticker_info (code,name,type,region,status) values ("%s","%s","ticker","CN",1)'%(code,name))
        #sql="insert ignore into qg_indicator_info (qg_code,name,category,need_process) values (%s,%s,'ticker',1)"
        #sql1="insert ignore into ticker_info (code,name,type,region,status) values (%s,%s,'stock','"+k+"',1)"
    else:
        #print(code,name)
        mysql.write_query('delete from qg_indicator_info where qg_code="%s"'%code)
        mysql.write_query("insert  into qg_indicator_info (qg_code,name,category,need_process) values ('%s','%s','edb',1)"%(code+'.WD',name))
        mysql.write_query('insert  into edb_info (code,name,wind_code) values ("%s","%s","%s")'%(code+'.WD',name,code))


Had problem on sql "insert  into qg_indicator_info (qg_code,name,category,need_process) values ('512480.SH','半导体ETF','ticker',1)" and rollback
(1062, "Duplicate entry '512480.SH' for key 'qg_indicator_info.PRIMARY'")
Had problem on sql "insert  into ticker_info (code,name,type,region,status) values ("512480.SH","半导体ETF","ticker","CN",1)" and rollback
(1062, "Duplicate entry '512480.SH' for key 'ticker_info.PRIMARY'")
Had problem on sql "insert  into qg_indicator_info (qg_code,name,category,need_process) values ('516620.SH','影视ETF','ticker',1)" and rollback
(1062, "Duplicate entry '516620.SH' for key 'qg_indicator_info.PRIMARY'")
Had problem on sql "insert  into ticker_info (code,name,type,region,status) values ("516620.SH","影视ETF","ticker","CN",1)" and rollback
(1062, "Duplicate entry '516620.SH' for key 'ticker_info.PRIMARY'")
Had problem on sql "insert  into qg_indicator_info (qg_code,name,category,need_process) values ('S2707426.WD','70城二手住宅价格指数:环比','edb',1)" and rollback
(1062, "Du

In [31]:
mysql

NameError: name 'mysql' is not defined